In [3]:
import sys
import os
home=os.getcwd()
sys.path.append(home+'/Functions')
from ShowDF import *
import numpy as np
import pandas as pd
import sys
import matplotlib.pyplot as plt

In [ ]:
from NeoPower import *
def ExtractPseudoMS2_from_MS1(DataSet,
                              sample_name,
                              out_folder,
                              min_mz=80,
                              max_mz=1000,
                              minInt=1e4,
                              ML=1,
                              MinDif=2.0038,
                              MaxDif=2.0046,
                              min_RT=0,
                              max_RT=np.inf,
                              theor_diff=2.0042449932959006):

    summary_rows = []
    pseudo_id = 0

    for scan_id, spectrum in enumerate(DataSet):
        if spectrum.getMSLevel() != ML:
            continue

        RT_seconds = spectrum.getRT()
        if not (min_RT < RT_seconds < max_RT):
            continue

        peaks = np.array(spectrum.get_peaks()).T
        peaks = peaks[(peaks[:,0] > min_mz) & (peaks[:,0] < max_mz) & (peaks[:,1] > minInt)]

        pairs = NeoPower(peaks,
                         MinDif = MinDif,
                         MaxDif = MaxDif)
        pairs = resolve_unique_pairs(pairs, theor_diff=theor_diff)

        for mz1, mz2, _, _, dmz, int1, int2 in pairs:
            pseudo_df = build_two_peak_pseudo_ms2(mz1, mz2, int1, int2)
            pseudo_df.to_csv(f"{out_folder}/{sample_name}mzML/{pseudo_id}.csv")

            summary_rows.append([pseudo_id, mz1, RT_seconds, mz2, dmz, int1 + int2])
            pseudo_id += 1

    return pd.DataFrame(summary_rows,
                        columns=["ms2_id", "mz", "RT", "mz_heavy", "dmz", "sumInt"])

In [4]:
from JoiningSummMS2 import *
ResultsFolder = home + '/Projects/O2O/ms2_spectra'
All_SummMS2Table, SamplesNames = JoiningSummMS2(ResultsFolder = ResultsFolder,
                                                mz_min = 100, 
                                                mz_max = 200,
                                                RT_min = 0,
                                                RT_max = 2000,
                                                ToReplace = 'mzML-ms2Summary.xlsx')


In [6]:
ShowDF(All_SummMS2Table[:10,:])

,0,1,2,3,4,5,6,7
0,2322,243.152,134.601,1703,3.85546e+06,0.0395446,9,0
1,2234,243.152,134.714,1713,5.09886e+06,0.0453376,10,1
2,2233,243.159,223.617,3472,4.94567e+06,0.0290257,1,2
3,2234,243.159,246.545,3872,5.51523e+06,0.0371287,1,3
4,2249,243.159,228.521,3563,7.48928e+06,0.0606791,2,4
5,2235,243.159,217.968,3347,1.23712e+07,0.0294316,0,5
6,2235,243.159,213.05,3287,1.24039e+07,0.0779744,1,6
7,2250,243.159,217.858,3376,1.86648e+07,0.0300854,2,7
8,2236,243.159,218.443,3381,1.61809e+07,0.0289003,1,8
9,2237,243.159,230.314,3588,5.92724e+06,0.0582262,1,9


In [23]:
from WorkLoadPlanning import *
EdgesMat = WorkLoadPlanning(All_SummMS2Table,
                            slices = 1,
                            mz_Tol = 2e-3)

np.float64(297.1597925508)

3

In [34]:

from ms2_SamplesAligment import *
ResultsFolder = home + '/Projects/O2O/ms2_spectra_Summ'
ms2Folder = home + '/Projects/O2O/ms2_spectra'
WorkingDirectory = home + '/Projects/O2O'

AlignedSamplesDF = ms2_SamplesAligment(ProjectName = 'O2O',
                                       All_SummMS2Table = All_SummMS2Table,
                                       EdgesMat = EdgesMat,
                                       SamplesNames = SamplesNames,
                                       RT_tol = 30,
                                       mz_Tol = 2e-3,
                                       cos_tol = 0.7,
                                       min_N_ms2_spectra = 3,
                                       ToReplace = 'mzML-ms2Summary.xlsx',
                                       ms2Folder = ms2Folder,
                                       ToAdd = 'mzML',
                                       Norm2One = True)

In [35]:
ShowDF(AlignedSamplesDF)

,median_mz(Da),IQR_mz(ppm),Q1_RT(s),median_RT(s),Q3_RT(s),N_samples,N_ms2-spectra,SamplingSamples,Q1_CosSim,median_CosSim,Q3_CosSim,min_CosSim,max_CosSim,min_mz,max_mz,min_RT(s),max_RT(s),feat_id,11011,11012,11013,11111,11121,11131,11211,11212,11213,11221,11222,11223,11231,11232,11233,12011,12012,12013,12111,12112,12113,12211,12212,12213,12221,12222,12223,12231,12232,12233,12311,12312,12313,13011,13012,13013,13111,13112,13113,13211,13212,13213,13221,13222,13223,13231,13232,13233,13311,13312,13313,21001,21002,21003
0,243.159,0.125505,217.913,217.968,218.206,3,3,0,0.965228,0.980212,0.99146,0.960233,0.99521,243.159,243.159,217.869,218.396,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,243.159,0.596147,212.755,212.792,212.921,3,3,0,0.995605,0.996892,0.998055,0.995175,0.998443,243.159,243.159,212.727,213.024,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,243.159,0.172569,220.741,233.706,245.705,4,4,0,0.952645,0.972812,0.978181,0.942718,0.987602,243.159,243.159,218.412,245.718,2,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,243.17,0.695789,106.583,106.99,107.544,8,8,0,0.969994,0.994077,0.9991,0.966281,0.999632,243.17,243.17,85.9279,109.13,3,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,1,1,0,0,0,0,0
4,243.17,0.627495,109.224,109.585,110.599,7,7,0,0.986511,0.991189,0.995361,0.984642,0.997544,243.17,243.17,108.031,111.614,4,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,1,0,0,1,0,0,0,0
5,243.17,0.376497,108.006,108.13,108.768,13,13,0,0.978566,0.988976,0.993407,0.972154,0.996532,243.17,243.17,106.915,109.663,5,1,1,1,0,0,0,0,0,0,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,1,0,0,0
6,244.085,0.522025,152.363,152.516,153.205,3,3,0,0.875739,0.939322,0.951831,0.854544,0.956001,244.085,244.086,152.242,153.756,6,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,244.096,0.546975,209.448,210.22,213.945,17,24,0,0.978809,0.988243,0.994324,0.964516,0.996771,244.096,244.096,208.566,215.432,7,1,1,1,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,1,1,0,0,0,0,0,0,0,0
8,244.096,0.449069,213.361,213.872,213.942,3,3,0,0.972573,0.974036,0.988184,0.972085,0.9929,244.096,244.096,212.952,213.999,8,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,244.124,3.20335,179.182,184.423,188.007,14,24,0,0.951124,0.979995,0.992899,0.907336,0.998386,244.123,244.126,156.509,191.34,9,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,1,1,1,0,0,0,0,0,0
